|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Reading through the table<h1>|
|<h2>Lecture:</h2>|<h1><b>The same arithmetic, with the keys scattered<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
# find the repo root, wherever this notebook was opened from
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

# Attention that reads through the page table

The arithmetic does not change. The addresses do.

Instead of `K[seq, kvh, pos]` you have a block table, a pool, and a lookup:

    physical = block_table[seq][pos // block_size]
    entry    = pool[physical][kvh][pos % block_size]

In [2]:
torch.manual_seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

S, H, KVH, D, L = 4, 8, 2, 64, 100
BLOCK = 16

K = torch.randn(S, KVH, L, D, device=dev)
V = torch.randn(S, KVH, L, D, device=dev)
q = torch.randn(S, H, D, device=dev)
print(f'{S} sequences, {L} tokens each, blocks of {BLOCK}')

4 sequences, 100 tokens each, blocks of 16


### Scatter the cache, on purpose

A real pool is shuffled: blocks are handed out as they come free, so a
sequence's blocks are in no particular order. Build it that way deliberately,
because a gather that quietly assumes order will pass a tidy test and fail in
production.

In [3]:
from tests.helpers import build_paged

kc, vc, block_tables, context_lens = build_paged(K, V, BLOCK)

print(f'pool:         {tuple(kc.shape)}   (blocks, kv_heads, block_size, head_dim)')
print(f'block table:  {tuple(block_tables.shape)}')
print(f'\nsequence 0 lives in blocks: {block_tables[0].tolist()}')
print(f'sequence 1 lives in blocks: {block_tables[1].tolist()}')
print('\nnot contiguous, not sorted, not adjacent. That is the point.')

pool:         (28, 2, 16, 64)   (blocks, kv_heads, block_size, head_dim)
block table:  (4, 7)

sequence 0 lives in blocks: [3, 14, 10, 17, 5, 0, 7]
sequence 1 lives in blocks: [21, 19, 18, 23, 25, 4, 2]

not contiguous, not sorted, not adjacent. That is the point.


### Gather, then do the same arithmetic

In [4]:
import math

def paged_attention(q, kc, vc, block_tables, context_lens, scale=None):
  S, H, D = q.shape
  KVH, BS = kc.shape[1], kc.shape[2]
  group   = H // KVH
  scale   = scale or 1.0/math.sqrt(D)
  out     = torch.empty_like(q)

  for s in range(S):
    n      = int(context_lens[s])
    blocks = block_tables[s, : (n + BS - 1)//BS].long()
    # (blocks, KVH, BS, D) -> (KVH, blocks*BS, D), then cut to n
    k = kc[blocks].permute(1,0,2,3).reshape(KVH, -1, D)[:, :n]
    v = vc[blocks].permute(1,0,2,3).reshape(KVH, -1, D)[:, :n]
    for h in range(H):
      kvh = h // group
      sc  = (k[kvh] @ q[s,h]) * scale
      out[s,h] = torch.softmax(sc, dim=0) @ v[kvh]
  return out

got = paged_attention(q, kc, vc, block_tables, context_lens)
print('output:', tuple(got.shape))

output: (4, 8, 64)


In [5]:
# the oracle: the same arithmetic on the ORIGINAL contiguous tensors
def reference_attention(q, K, V):
  S, H, D = q.shape
  group   = H // K.shape[1]
  out = torch.empty_like(q)
  for s in range(S):
    for h in range(H):
      sc = (K[s, h//group] @ q[s,h]) / math.sqrt(D)
      out[s,h] = torch.softmax(sc, dim=0) @ V[s, h//group]
  return out

want = reference_attention(q, K, V)
print('max difference:', (got - want).abs().max().item())
print('\nthe gather found every byte, in the right order, from a shuffled pool')

max difference: 0.0

the gather found every byte, in the right order, from a shuffled pool


### And the masking trap

Blocks are recycled. A slot past `context_len` holds whatever the last
sequence left there. Score it and you get a plausible, wrong answer.

In [6]:
kc2, vc2 = kc.clone(), vc.clone()
for s in range(S):
  for b in range(block_tables.shape[1]):
    phys = int(block_tables[s,b])
    for off in range(BLOCK):
      if b*BLOCK + off >= L:
        kc2[phys,:,off] = 999.0       # someone else's tokens
        vc2[phys,:,off] = 999.0

got2 = paged_attention(q, kc2, vc2, block_tables, context_lens)
print('unchanged after poisoning the unused slots:',
      torch.allclose(got, got2, atol=1e-5))
print('\n(this passes because the gather cuts to n. Forget that and it will not.)')

unchanged after poisoning the unused slots: True

(this passes because the gather cuts to n. Forget that and it will not.)


# What the gather cost you

In [7]:
if dev == 'cuda':
  S, H, KVH, D, L = 32, 16, 8, 128, 512
  K = torch.randn(S, KVH, L, D, device=dev, dtype=torch.float16)
  V = torch.randn(S, KVH, L, D, device=dev, dtype=torch.float16)
  q = torch.randn(S, H, D, device=dev, dtype=torch.float16)
  kc, vc, bt, ctx = build_paged(K, V, BLOCK)

  paged = cudalib.bench_ms(lambda: paged_attention(q, kc, vc, bt, ctx), iters=5, warmup=2)
  dense = cudalib.bench_ms(lambda: F.scaled_dot_product_attention(
            q.unsqueeze(2), K.repeat_interleave(H//KVH,1), V.repeat_interleave(H//KVH,1)),
            iters=20, warmup=5)

  print(f'contiguous SDPA:  {dense:8.3f} ms')
  print(f'paged, in python: {paged:8.3f} ms')
  print(f'\nyou just paid {paged/dense:.0f}x for the memory you saved')

contiguous SDPA:     1.125 ms
paged, in python:   19.752 ms

you just paid 18x for the memory you saved


### That number is the bill for stage 06

Paging bought about ten times the sequences resident. It cost a factor like
that in attention speed, because the implementation above is a **Python loop
over sequences** doing one gather and one matmul each.

Two separate problems live in that sentence:

- one kernel launch per sequence per head, when there should be one launch
- the gathered K and V are **materialised**, so every byte makes a round trip
  to HBM before anything looks at it

Both go away in one move, and this function becomes the oracle you check it
against forever.

    ./vc guide 8